In [1]:
import sys
sys.path.append('..')

import torch
from src import models, data, lens, functional
from src.utils import experiment_utils
from baukit import Menu, show

In [3]:
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
mt = models.load_model("gptj", device=device, fp16=True)
print(f"dtype: {mt.model.dtype}, device: {mt.model.device}, memory: {mt.model.get_memory_footprint()}")

Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

GPTJForCausalLM LOAD REPORT from: EleutherAI/gpt-j-6B
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...27}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...27}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/930 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

dtype: torch.float16, device: cuda:0, memory: 12109105600


In [13]:
dataset = data.load_dataset()

relation_names = [r.name for r in dataset.relations]
# relation_options = Menu(choices = relation_names, value = relation_names)
# show(relation_options) # !caution: tested in a juputer-notebook. baukit visualizations are not supported in vscode.

In [14]:
relation_names

['characteristic gender',
 'univ degree gender',
 'name birthplace',
 'name gender',
 'name religion',
 'occupation age',
 'occupation gender',
 'fruit inside color',
 'fruit outside color',
 'object superclass',
 'substance phase of matter',
 'task person type',
 'task done by tool',
 'word sentiment',
 'work location',
 'city in country',
 'company CEO',
 'company hq',
 'country capital city',
 'country currency',
 'country language',
 'country largest city',
 'food from country',
 'landmark in country',
 'landmark on continent',
 'person lead singer of band',
 'person father',
 'person mother',
 'person native language',
 'person occupation',
 'person plays instrument',
 'person sport position',
 'plays pro sport',
 'person university',
 'pokemon evolution',
 'president birth year',
 'president election year',
 'product by company',
 'star constellation name',
 'superhero archnemesis',
 'superhero person',
 'adjective antonym',
 'adjective comparative',
 'adjective superlative',
 'v

In [35]:
# select one from the above
relation_name='country capital city'

# relation_name = relation_options.value
# if isinstance(relation_name, (list, tuple)):
#     relation_name = relation_name[0]

relation = dataset.filter(relation_names=[relation_name])[0]
print(f"{relation.name} -- {len(relation.samples)} samples")
print("------------------------------------------------------")

experiment_utils.set_seed(12345) # set seed to a constant value for sampling consistency
train, test = relation.split(5)
print("\n".join([sample.__str__() for sample in train.samples]))

country capital city -- 24 samples
------------------------------------------------------
China -> Beijing
Japan -> Tokyo
Italy -> Rome
Brazil -> Bras\u00edlia
Turkey -> Ankara


In [36]:
################### hparams ###################
layer = 5
beta = 2.5
###############################################

In [37]:
from src.operators import JacobianIclMeanEstimator

estimator = JacobianIclMeanEstimator(
    mt = mt, 
    h_layer = layer,
    beta = beta
)
operator = estimator(
    relation.set(
        samples=train.samples, 
    )
)

relation has > 1 prompt_templates, will use first (The capital city of {} is)


# Checking $faithfulness$

In [38]:
test = functional.filter_relation_samples_based_on_provided_fewshots(
    mt=mt, test_relation=test, prompt_template=operator.prompt_template, batch_size=4
)

In [39]:
sample = test.samples[0]
print(sample)
operator(subject = sample.subject).predictions

Argentina -> Buenos Aires


[PredictedToken(token='\n', prob=0.2518749237060547),
 PredictedToken(token=' ', prob=0.18141870200634003),
 PredictedToken(token=' ...', prob=0.12665066123008728),
 PredictedToken(token=' Buenos', prob=0.057085927575826645),
 PredictedToken(token=' the', prob=0.03862627223134041)]

In [40]:
hs_and_zs = functional.compute_hs_and_zs(
    mt = mt,
    prompt_template = operator.prompt_template,
    subjects = [sample.subject],
    h_layer= operator.h_layer,
)
h = hs_and_zs.h_by_subj[sample.subject]

## Approximating LM computation $F$ as an affine transformation

### $$ F(\mathbf{s}, c_r) \approx \beta \, W_r \mathbf{s} + b_r $$

In [41]:
z = operator.beta * (operator.weight @ h) + operator.bias

lens.logit_lens(
    mt = mt,
    h = z,
    get_proba = True
)

([('\n', 0.252),
  (' ', 0.181),
  (' ...', 0.127),
  (' Buenos', 0.057),
  (' the', 0.039),
  ('...', 0.036),
  (' Bras', 0.02),
  ('\\', 0.016),
  (' (', 0.016),
  (' Rome', 0.015)],
 {})

In [42]:
correct = 0
wrong = 0
for sample in test.samples:
    predictions = operator(subject = sample.subject).predictions
    known_flag = functional.is_nontrivial_prefix(
        prediction=predictions[0].token, target=sample.object
    )
    print(f"{sample.subject=}, {sample.object=}, ", end="")
    print(f'predicted="{functional.format_whitespace(predictions[0].token)}", (p={predictions[0].prob}), known=({functional.get_tick_marker(known_flag)})')
    
    correct += known_flag
    wrong += not known_flag
    
faithfulness = correct/(correct + wrong)

print("------------------------------------------------------------")
print(f"Faithfulness (@1) = {faithfulness}")
print("------------------------------------------------------------")

sample.subject='Argentina', sample.object='Buenos Aires', predicted="\n", (p=0.2518749237060547), known=(✗)
sample.subject='Australia', sample.object='Canberra', predicted=" ...", (p=0.17288744449615479), known=(✗)
sample.subject='Canada', sample.object='Ottawa', predicted=" ...", (p=0.12184084951877594), known=(✗)
sample.subject='Chile', sample.object='Santiago', predicted="\n", (p=0.3059007227420807), known=(✗)
sample.subject='Colombia', sample.object='Bogot\\u00e1', predicted="\n", (p=0.3171284794807434), known=(✗)
sample.subject='Egypt', sample.object='Cairo', predicted="\n", (p=0.22527247667312622), known=(✗)
sample.subject='France', sample.object='Paris', predicted=" Paris", (p=0.8408417105674744), known=(✓)
sample.subject='Germany', sample.object='Berlin', predicted=" Berlin", (p=0.3902198076248169), known=(✓)
sample.subject='India', sample.object='New Delhi', predicted=" New", (p=0.13692353665828705), known=(✓)
sample.subject='Mexico', sample.object='Mexico City', predicted=" .

# $causality$

In [43]:
################### hparams ###################
rank = 100
###############################################

In [44]:
experiment_utils.set_seed(12345) # set seed to a constant value for sampling consistency
test_targets = functional.random_edit_targets(test.samples)

## setup

In [45]:
source

RelationSample(subject='bloggers', object='young')

In [46]:
test_targets

{RelationSample(subject='Argentina', object='Buenos Aires'): RelationSample(subject='Saudi Arabia', object='Riyadh'),
 RelationSample(subject='Australia', object='Canberra'): RelationSample(subject='Argentina', object='Buenos Aires'),
 RelationSample(subject='Canada', object='Ottawa'): RelationSample(subject='Nigeria', object='Abuja'),
 RelationSample(subject='Chile', object='Santiago'): RelationSample(subject='Peru', object='Lima'),
 RelationSample(subject='Colombia', object='Bogot\\u00e1'): RelationSample(subject='Germany', object='Berlin'),
 RelationSample(subject='Egypt', object='Cairo'): RelationSample(subject='Mexico', object='Mexico City'),
 RelationSample(subject='France', object='Paris'): RelationSample(subject='Saudi Arabia', object='Riyadh'),
 RelationSample(subject='Germany', object='Berlin'): RelationSample(subject='Egypt', object='Cairo'),
 RelationSample(subject='India', object='New Delhi'): RelationSample(subject='Peru', object='Lima'),
 RelationSample(subject='Mexico',

In [48]:
source = test.samples[-2]
target = test_targets[source]

f"Changing the mapping ({source}) to ({source.subject} -> {target.object})"

'Changing the mapping (United States -> Washington D.C.) to (United States -> Ottawa)'

### Calculate $\Delta \mathbf{s}$ such that $\mathbf{s} + \Delta \mathbf{s} \approx \mathbf{s}'$

<p align="center">
    <img align="center" src="causality-crop.png" style="width:80%;"/>
</p>

Under the relation $r =\, $*plays the instrument*, and given the subject $s =\, $*Miles Davis*, the model will predict $o =\, $*trumpet* **(a)**; and given the subject $s' =\, $*Cat Stevens*, the model will now predict $o' =\, $*guiter* **(b)**. 

If the computation from $\mathbf{s}$ to $\mathbf{o}$ is well-approximated by $operator$ parameterized by $W_r$ and $b_r$ **(c)**, then $\Delta{\mathbf{s}}$ **(d)** should tell us the direction of change from $\mathbf{s}$ to $\mathbf{s}'$. Thus, $\tilde{\mathbf{s}}=\mathbf{s}+\Delta\mathbf{s}$ would be an approximation of $\mathbf{s}'$ and patching $\tilde{\mathbf{s}}$ in place of $\mathbf{s}$ should change the prediction to $o'$ = *guitar* 

In [49]:
def get_delta_s(
    operator, 
    source_subject, 
    target_subject,
    rank = 100,
    fix_latent_norm = None, # if set, will fix the norms of z_source and z_target
):
    w_p_inv = functional.low_rank_pinv(
        matrix = operator.weight,
        rank=rank,
    )
    hs_and_zs = functional.compute_hs_and_zs(
        mt = mt,
        prompt_template = operator.prompt_template,
        subjects = [source_subject, target_subject],
        h_layer= operator.h_layer,
        z_layer=-1,
    )

    z_source = hs_and_zs.z_by_subj[source_subject]
    z_target = hs_and_zs.z_by_subj[target_subject]
    
    z_source *= fix_latent_norm / z_source.norm() if fix_latent_norm is not None else 1.0
    z_target *= z_source.norm() / z_target.norm() if fix_latent_norm is not None else 1.0

    delta_s = w_p_inv @  (z_target.squeeze() - z_source.squeeze())

    return delta_s, hs_and_zs

delta_s, hs_and_zs = get_delta_s(
    operator = operator,
    source_subject = source.subject,
    target_subject = target.subject,
    rank = rank
)

In [50]:
import baukit

def get_intervention(h, int_layer, subj_idx):
    def edit_output(output, layer):
        if(layer != int_layer):
            return output
        functional.untuple(output)[:, subj_idx] = h 
        return output
    return edit_output

prompt = operator.prompt_template.format(source.subject)

h_index, inputs = functional.find_subject_token_index(
    mt=mt,
    prompt=prompt,
    subject=source.subject,
)

h_layer, z_layer = models.determine_layer_paths(model = mt, layers = [layer, -1])

with baukit.TraceDict(
    mt.model, layers = [h_layer, z_layer],
    edit_output=get_intervention(
#         h = hs_and_zs.h_by_subj[source.subject],         # let the computation proceed as usual
        h = hs_and_zs.h_by_subj[source.subject] + delta_s, # replace s with s + delta_s
        int_layer = h_layer, 
        subj_idx = h_index
    )
) as traces:
    outputs = mt.model(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
    )

lens.interpret_logits(
    mt = mt, 
    logits = outputs.logits[0][-1], 
    get_proba=True
)

[(' Ottawa', 0.766),
 (' Toronto', 0.073),
 (' Mont', 0.03),
 (' Montreal', 0.02),
 (' Canada', 0.02),
 (' Quebec', 0.01),
 ('\n', 0.007),
 (' London', 0.006),
 (' Victoria', 0.006),
 (' ...', 0.004)]

## Measuring causality

In [51]:
from src.editors import LowRankPInvEditor

svd = torch.svd(operator.weight.float())
editor = LowRankPInvEditor(
    lre=operator,
    rank=rank,
    svd=svd,
)

In [52]:
# precomputing latents to speed things up
hs_and_zs = functional.compute_hs_and_zs(
    mt = mt,
    prompt_template = operator.prompt_template,
    subjects = [sample.subject for sample in test.samples],
    h_layer= operator.h_layer,
    z_layer=-1,
    batch_size = 2
)

success = 0
fails = 0

for sample in test.samples:
    target = test_targets.get(sample)
    assert target is not None
    edit_result = editor(
        subject = sample.subject,
        target = target.subject
    )
    
    success_flag = functional.is_nontrivial_prefix(
        prediction=edit_result.predicted_tokens[0].token, target=target.object
    )
    
    print(f"Mapping {sample.subject} -> {target.object} | edit result={edit_result.predicted_tokens[0]} | success=({functional.get_tick_marker(success_flag)})")
    
    success += success_flag
    fails += not success_flag
    
causality = success / (success + fails)

print("------------------------------------------------------------")
print(f"Causality (@1) = {causality}")
print("------------------------------------------------------------")

Mapping Argentina -> Riyadh | edit result= Riyadh (p=0.736) | success=(✓)
Mapping Australia -> Buenos Aires | edit result= Buenos (p=0.898) | success=(✓)
Mapping Canada -> Abuja | edit result= Abu (p=0.694) | success=(✓)
Mapping Chile -> Lima | edit result= Lima (p=0.766) | success=(✓)
Mapping Colombia -> Berlin | edit result= Berlin (p=0.970) | success=(✓)
Mapping Egypt -> Mexico City | edit result= Mexico (p=0.975) | success=(✓)
Mapping France -> Riyadh | edit result= Riyadh (p=0.752) | success=(✓)
Mapping Germany -> Cairo | edit result= Cairo (p=0.947) | success=(✓)
Mapping India -> Lima | edit result= Lima (p=0.647) | success=(✓)
Mapping Mexico -> Santiago | edit result= Santiago (p=0.849) | success=(✓)
Mapping Nigeria -> Riyadh | edit result= Riyadh (p=0.739) | success=(✓)
Mapping Pakistan -> New Delhi | edit result= New (p=0.742) | success=(✓)
Mapping Peru -> Caracas | edit result= Car (p=0.278) | success=(✓)
Mapping Russia -> Cairo | edit result= Cairo (p=0.967) | success=(✓)
Ma